# Assignment 1, Task A: Classification problem.

## The data:
In this QSAR exercise, the mutagenicity of various molecules is to be investigated. The dataset in use is the Ames Mutagenicity Dataset for Multi-Task learning accessed via the PyTDC library, essentially as also provided here: https://huggingface.co/datasets/scikit-fingerprints/TDC_ames. Columns have been renamed for enhanced clarity.

The dataset gives the overal mutagenicity (1 = mutagen) of various drugs (simply represented as their SMILES string). From the SMILES strings, molecular fingerprints can be generated as molecular descriptors.

## The tasks:
1) Inspect the data and clean if needed. Adhere to good practices!
2) Calculate the fingerprints (partial snippet provided) and create a feature matrix X and a target vector y
3) Then four different models should be trained on the fingerprints and evaluated according to accuracy and their roc-auc score to compare their performance. For each model, additionally, the overfitting needs to be addressed.

These four models have to be compared:
- `KNeighborsClassifier`: choose a suitable number of neighbors
- `DecisionTreeClassifier`: use a random_state
- `RandomForestClassifier`: use a random_state and a slightly bigger forest (e.g. 200 trees)
- `GradientBoostingClassifier`: use a random_state

Other than the stated parameters, the models can be mostly used as provided by `scikit`. No hyperparameter tuning needs to be performed, no CV necessary.

4) Conclusion and discussion: Provide answers to the questions.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, ConfusionMatrixDisplay

In [5]:
df = pd.read_csv("../assignments/assignment01/ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


## 1. Inspect and clean the data
- Gain some overview of the data and assess NaNs and duplicates and clean if needed.
- Inspect the class balance!

In [7]:
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7278 entries, 0 to 7277
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   drug_id       7278 non-null   object
 1   smiles        7278 non-null   object
 2   mutagenicity  7278 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 170.7+ KB


## 2. Create fingerprints from the Smiles
The partial snippet for MorganFingerprints can be used. Note that instead of a dataframe, the function will produce a np.array, which will be written into a list. From this you can create the feature matrix and the target vector. Inspect the shape of the arrays!

In [10]:
def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fp = mfpgen.GetFingerprint(mol)
    return np.array(fp)

# Convert to fingerprints
fps = []
valid_labels = []

for smiles, label in zip(df["smiles"], df["mutagenicity"]):
    fp = smiles_to_fp(smiles)
    if fp is not None:
        fps.append(fp)
        valid_labels.append(label)


In [18]:
X = np.array(fps)
y= np.array(valid_labels)

print(X.shape, y.shape)

(7278, 2048) (7278,)


## 3. Train the models
Use a classic train-test split of 0.2 including a random seed and `stratify`. For training and predicting labels, take note of the time the process takes for each model (does not necessarily have to be coded, can also be estimated). Make sure to predict labels for both training and test splits in order to identify overfitting. Use the accuracy and roc-auc as metrics for evaluation.

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,stratify=y)

In [ ]:
from sklearn.model_selection import cross_val_score

k_vals = range(1,51)
mean_scores = []

for k in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=None, scoring='accuracy')
    mean_scores.append(scores.mean())
#plt.plot(k_vals,mean_scores)
#plt.show()




0.7789394274589619


In [ ]:
print(mean_scores.index(max(mean_scores))+1)
#chose a k value of 3

3


In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_train_pred_k = knn.predict(X_train)
y_test_pred_k = knn.predict(X_test)

train_acc_k = accuracy_score(y_train,y_train_pred_k)
test_acc_k = accuracy_score(y_test,y_test_pred_k)

train_roc_k = roc_auc_score(y_train,y_train_pred_k)
test_roc_k = roc_auc_score(y_test,y_test_pred_k)

print(f"Train accuracy: {train_acc_k:.3f}, Test_accuracy: {test_acc_k:.3f}")
print(f"Train ROC AUC: {train_roc_k:.3f}, Test ROC AUC: {test_roc_k:.3f}")


Train accuracy: 0.896, Test_accuracy: 0.782
Train ROC AUC: 0.895, Test ROC AUC: 0.780


In [42]:

tree = DecisionTreeClassifier()
tree.fit(X_train,y_train)

y_train_pred_t = tree.predict(X_train)
y_test_pred_t = tree.predict(X_test)

train_acc_t = accuracy_score(y_train,y_train_pred_t)
test_acc_t = accuracy_score(y_test,y_test_pred_t)

train_roc_t = roc_auc_score(y_train,y_train_pred_t)
test_roc_t = roc_auc_score(y_test,y_test_pred_t)

print(f"Train accuracy: {train_acc_t:.3f}, Test_accuracy: {test_acc_t:.3f}")
print(f"Train ROC AUC: {train_roc_t:.3f}, Test ROC AUC: {test_roc_t:.3f}")


Train accuracy: 0.999, Test_accuracy: 0.764
Train ROC AUC: 0.999, Test ROC AUC: 0.763


In [43]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

y_train_pred_r = rf.predict(X_train)
y_test_pred_r = rf.predict(X_test)

train_acc_r = accuracy_score(y_train,y_train_pred_r)
test_acc_r = accuracy_score(y_test,y_test_pred_r)

train_roc_r = roc_auc_score(y_train,y_train_pred_r)
test_roc_r = roc_auc_score(y_test,y_test_pred_r)

print(f"Train accuracy: {train_acc_r:.3f}, Test_accuracy: {test_acc_r:.3f}")
print(f"Train ROC AUC: {train_roc_r:.3f}, Test ROC AUC: {test_roc_r:.3f}")


Train accuracy: 0.999, Test_accuracy: 0.815
Train ROC AUC: 0.999, Test ROC AUC: 0.813


In [44]:
gb = GradientBoostingClassifier()
gb.fit(X_train,y_train)

y_train_pred_g = gb.predict(X_train)
y_test_pred_g = gb.predict(X_test)

train_acc_g = accuracy_score(y_train,y_train_pred_g)
test_acc_g = accuracy_score(y_test,y_test_pred_g)

train_roc_g = roc_auc_score(y_train,y_train_pred_g)
test_roc_g = roc_auc_score(y_test,y_test_pred_g)

print(f"Train accuracy: {train_acc_g:.3f}, Test_accuracy: {test_acc_g:.3f}")
print(f"Train ROC AUC: {train_roc_g:.3f}, Test ROC AUC: {test_roc_g:.3f}")


Train accuracy: 0.819, Test_accuracy: 0.757
Train ROC AUC: 0.819, Test ROC AUC: 0.755


## 4. Conclusion and discussion
- Which model performed the best?
- Which was the most time efficient?
- Which model showed the wors overfitting?
- Why does ensemble learning outperform a single tree?
- Why does KNN perform well in high-dimensional fingerprint space?
- What does ROC-AUC tell us that accuracy does not?

1. The random forest model performed the best. It had both the highest accuracy and ROC & AUC values. It provided the best generalisation.
2. The most time efficient model was the normal decision tree. The KNN was faster but cross validation had to be done to find a suitable k value which made the process a lot slower
3. The normal tree showed the worst overfitting. Cross validation would be good as well to limit the tree depth
4. Ensembles are more stable and generalise better due to their decorrelated variations -> less prone to overfitting
5. In high dimensions, it is a lot easier to cluster things together than trying to form clusters in overlapping points in 2 dimensions => More separability
6. In a small patient sample, our accuracy might be very high in a bad model because most of the patients are healthy and it can just diagnose every person as healthy and still guess almost all of them correctly. ROC looks at both precision and recall. It plots the true positive rate vs the false positive rate which leads to much more stable results